In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import datetime

# Outlier Detection and Data Quality

Extreme values in price, square footage, or days on market can distort market averages and trends. I will implement a statistical method to identify and remove these records.

## Method: Interquartile Range (IQR)

The IQR method removes records that fall outside a defined statistical range:

```
Q1 = df['ClosePrice'].quantile(0.25)
Q3 = df['ClosePrice'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df = df[(df['ClosePrice'] >= lower) & (df['ClosePrice'] <= upper)]
```

These are the key numeric fields: ClosePrice, ListPrice, OriginalListPrice, LivingArea, LotSizeSquareFeet, BedroomsTotal, BathroomsTotalInteger, DaysOnMarket, and YearBuilt. I will use the 1.5*IQR rule to flag outliers. I will be unable to perform the IQR method on `listing` data because that dataset does not have some of the key numeric fields mentioned.

In [87]:
sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week6/sold_clean_with_metrics.csv')
key_numeric_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

/var/folders/q8/sdky6zrj6y52rcdpgrsws4vm0000gn/T/ipykernel_13633/2844566093.py:1: DtypeWarning: Columns (0,1,7,58,61) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week6/sold_clean_with_metrics.csv')


In [88]:
sold.size

31812560

In [89]:
for field in key_numeric_fields:
    print(f"Median of {field}: {sold[field].median()}")

Median of ClosePrice: 815000.0
Median of ListPrice: 800000.0
Median of OriginalListPrice: 823652.0
Median of LivingArea: 1646.0
Median of LotSizeSquareFeet: 7278.0
Median of BedroomsTotal: 3.0
Median of BathroomsTotalInteger: 2.0
Median of DaysOnMarket: 19.0
Median of YearBuilt: 1979.0


In [90]:
# drop rows with negative values in each key numeric field
for field in key_numeric_fields:
    sold_nonnegative = sold[sold[field] >= 0].copy()

Now clear values that are extreme but clearly not outliers (e.g., data entry error). Use percentiles identified from Week 2's numeric distribution review.

In [91]:
percentile_caps = {
    'ClosePrice': 0.99,
    'ListPrice': 0.99,
    'OriginalListPrice': 0.99,
    'LivingArea': 0.99,
    'LotSizeSquareFeet': 0.95,
    'BedroomsTotal': 0.999,
    'BathroomsTotalInteger': 0.999,
    'DaysOnMarket': 0.99,
    'YearBuilt': 0.999
}

sold_capped = sold_nonnegative.copy()

for field, percentile in percentile_caps.items():
    upper_bound = sold_capped[field].quantile(percentile)
    sold_capped[field] = sold_capped[field].clip(upper=upper_bound)

In [92]:
sold_filtered = sold_capped.copy()

for field in key_numeric_fields:
    q1 = sold_filtered[field].quantile(0.25)
    q3 = sold_filtered[field].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    sold_filtered[field] = sold_filtered[field][(sold_filtered[field] >= lower_bound) & (sold_filtered[field] <= upper_bound)]

In [93]:
(sold_filtered[field] < lower_bound) | (sold_filtered[field] > upper_bound)

0         False
1         False
2         False
3         False
4         False
          ...  
397652    False
397653    False
397654    False
397655    False
397656    False
Name: YearBuilt, Length: 397287, dtype: bool

In [94]:
sold_filtered.size

31782960

In [95]:
for field in key_numeric_fields:
    print(f"Median of {field}: {sold_filtered[field].median()}")

Median of ClosePrice: 775000.0
Median of ListPrice: 775000.0
Median of OriginalListPrice: 785000.0
Median of LivingArea: 1607.0
Median of LotSizeSquareFeet: 6627.0
Median of BedroomsTotal: 3.0
Median of BathroomsTotalInteger: 2.0
Median of DaysOnMarket: 16.0
Median of YearBuilt: 1979.0


In [96]:
sold_filtered.to_csv('/Users/olee1232/Documents/IDXWork/Week7/sold_no_outliers.csv', index=False)

### Why This Matters

Outliers, such as a $50 million estate or a $10,000 distressed sale. can skew median prices and misrepresent the typical market. However, do not blindly remove outliers. Instead, use a tiered approach: flag extreme values first, apply business rules (e.g., ClosePrice <= 0 is always invalid), and create a separate filtered analysis dataset rather than permanently deleting records. Use percentiles alongside IQR to guide decisions.